# Auction Results Fact Table — Gold Layer (`company_db.gold.auction_results_fact`)

This notebook builds and maintains the **Auction Results Fact** table in the Gold layer of the Company data warehouse. It is the primary fact table for lot-level auction performance — covering pricing, estimates, sold/unsold status, and buyer information.

---

## What This Notebook Does

1. Creates a staging view (`final_auction_results`) that joins Silver auction results with winning bidder records and the Gold Users Dimension to resolve surrogate keys
2. Previews the staged data for validation
3. Initialises the `auction_results_fact` Delta table if it does not already exist
4. Merges staged records into the fact table (insert new lots, update existing ones)

---

## Source Tables

| Table | Layer | Description |
|---|---|---|
| `company_db.silver.auctionresults` | Silver | One row per lot per auction — pricing, estimates, and outcome |
| `company_db.silver.winningbidders` | Silver | Winning bidder per lot (joined on `lot_number + auction_id`) |
| `company_db.gold.users_dim` | Gold | User dimension — provides the `user_dim_key` surrogate FK |

## Target Table

`company_db.gold.auction_results_fact`  
Stored in Azure Data Lake Storage Gen2: `abfss://auction@company.dfs.core.windows.net/gold/auction_results_fact`

---

## MERGE Key

Records are matched on the **composite natural key**: `auction_id + lot_id`  
This enforces one fact row per lot per auction.

---

## Column Reference

| Column | Type | Description |
|---|---|---|
| `user_dim_key` | BIGINT | **Foreign key** → `company_db.gold.users_dim.user_dim_key` (NULL if lot was unsold) |
| `auction_id` | INTEGER | Auction identifier |
| `user_id` | INTEGER | Winning bidder’s source-system user ID (NULL if unsold) |
| `consignor_id` | DOUBLE | Consignor who submitted the lot for sale |
| `lot_id` | DOUBLE | Unique lot identifier |
| `lot_number` | DOUBLE | Lot number within the auction catalogue |
| `quantity` | INTEGER | Number of units in the lot |
| `startingbid` | DOUBLE | Opening bid amount |
| `lowestimate` | DOUBLE | Low pre-sale estimate |
| `highestimate` | DOUBLE | High pre-sale estimate |
| `midestimate` | DOUBLE | Mid-point pre-sale estimate |
| `hammerprice` | DOUBLE | Final hammer price (0 if the lot did not sell) |
| `commissionamount` | DOUBLE | Seller’s commission charged on the hammer price |
| `buyerspremium` | DOUBLE | Buyer’s premium added on top of the hammer price |
| `pricerealized` | DOUBLE | Total amount paid by the buyer (`hammerprice + buyerspremium`) |
| `favoritescount` | DOUBLE | Number of times the lot was added to a user’s favourites |
| `sold_flag` | INTEGER | `1` = sold, `0` = passed / unsold |
| `created_at` | TIMESTAMP | Timestamp when the row was first inserted |
| `updated_at` | TIMESTAMP | Timestamp of the last MERGE update |

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

---
## Step 1 · Create Staging View (`final_auction_results`)

Creates (or replaces) a **staging view** by joining three tables:

| Join | Type | Key |
|---|---|---|
| `silver.auctionresults` → `silver.winningbidders` | LEFT JOIN | `lot_number + auction_id` |
| `silver.winningbidders` → `gold.users_dim` | LEFT JOIN | `user_id` |

Both joins are LEFT JOINs so that:
- Unsold lots (no winning bidder) are retained with `user_id = NULL`
- Lots with a winning bidder not yet in the dimension are retained with `user_dim_key = NULL`

A derived `sold_flag` column is computed: **`1`** if `pricerealized > 0`, **`0`** otherwise.

> **Note:** This is a logical view — no data is written at this step.

In [ ]:
spark.sql(f"""
CREATE VIEW IF NOT EXISTS {catalog}.gold.final_auction_results AS
(select 
ar.auction_id,
wb.user_id,
u.user_dim_key,
dl.lot_dim_key,
cd.consignor_dim_key,
ar.consignor_id,
ar.lot_id,
ar.lot_number,
ar.quantity,
ar.startingbid,
ar.lowestimate,
ar.highestimate,
ar.midestimate,
ar.hammerprice,
ar.commissionamount,
ar.buyerspremium,
ar.pricerealized,
ar.favoritescount,
ar.start_date,
ar.end_date,
CASE WHEN ar.pricerealized = 0 THEN 0 ELSE 1 END AS sold_flag
from 
{catalog}.silver.auctionresults as ar 
left join {catalog}.silver.winningbidders as wb
ON wb.lot_number = ar.lot_number AND 
wb.auction_id = ar.auction_id
LEFT JOIN {catalog}.gold.users_dim as u
ON wb.user_id = u.user_id
LEFT JOIN {catalog}.gold.lots_dim as dl
ON (ar.start_date = dl.start_date AND ar.lot_number = dl.lot_number)
LEFT JOIN {catalog}.gold.consignor_dim as cd
ON ar.consignor_id = cd.consignor_id )""")

In [ ]:
auction_results = spark.sql(f""" 
(select 
ar.auction_id,
wb.user_id,
u.user_dim_key,
dl.lot_dim_key,
cd.consignor_dim_key,
ar.consignor_id,
ar.lot_id,
ar.lot_number,
ar.quantity,
ar.startingbid,
ar.lowestimate,
ar.highestimate,
ar.midestimate,
ar.hammerprice,
ar.commissionamount,
ar.buyerspremium,
ar.pricerealized,
ar.favoritescount,
ar.start_date,
ar.end_date,
CASE WHEN ar.pricerealized = 0 THEN 0 ELSE 1 END AS sold_flag
from 
{catalog}.silver.auctionresults as ar 
left join {catalog}.silver.winningbidders as wb
ON wb.lot_number = ar.lot_number AND 
wb.auction_id = ar.auction_id
LEFT JOIN {catalog}.gold.users_dim as u
ON wb.user_id = u.user_id
LEFT JOIN {catalog}.gold.lots_dim as dl
ON (ar.start_date = dl.start_date AND ar.lot_number = dl.lot_number)
LEFT JOIN {catalog}.gold.consignor_dim as cd
ON ar.consignor_id = cd.consignor_id )
""")

In [ ]:
auction_results = auction_results.dropDuplicates(['auction_id','lot_id'])

In [ ]:
auction_results.createOrReplaceTempView("auction_results")

---
## Step 2 · Initialize Gold Fact Table

Creates `company_db.gold.auction_results_fact` as a **Delta table** if it does not already exist. On subsequent runs this is a no-op — no existing data is affected.

Granularity: **one row per `auction_id` × `lot_id` pair** (enforced by the MERGE key in Step 3).

The table is backed by ADLS Gen2 at `gold/auction_results_fact` and supports full ACID transactions (MERGE, time travel, schema evolution).

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.auction_results_fact
(
  user_dim_key BIGINT,
  lot_dim_key BIGINT,
  consignor_dim_key BIGINT,
  auction_id INTEGER,
  user_id INTEGER,
  consignor_id DOUBLE,
  lot_id DOUBLE,
  lot_number DOUBLE,
  quantity INTEGER,
  startingbid DOUBLE,
  lowestimate DOUBLE,
  highestimate DOUBLE,
  midestimate DOUBLE,
  hammerprice DOUBLE,
  commissionamount DOUBLE,
  buyerspremium DOUBLE,
  pricerealized DOUBLE,
  favoritescount DOUBLE,
  start_date DATE,
  end_date DATE,
  sold_flag INTEGER,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/gold/auction_results_fact';""")

---
## Step 3 · MERGE into Gold Fact Table

Applies staged records from `final_auction_results` into `auction_results_fact` using a two-condition match key: **`auction_id + lot_id`**.

| Condition | Action | Notes |
|---|---|---|
| MATCHED | **UPDATE** all pricing, bidder, and status fields + `updated_at = current_timestamp()` | Keeps hammer prices, estimates, and sold status current as auctions close |
| NOT MATCHED | **INSERT** all columns with `created_at = current_timestamp()`, `updated_at = current_timestamp()` | Adds new lot results |

> `user_dim_key` is only written on **INSERT**. It is intentionally not updated on MATCH to preserve the stable FK link to the dimension.

In [ ]:
%sql
MERGE INTO company_db.gold.auction_results_fact AS target
USING auction_results  AS source
ON target.auction_id = source.auction_id
   AND target.lot_id = source.lot_id
   AND target.start_date = source.start_date
WHEN MATCHED THEN
UPDATE SET
  target.user_id = source.user_id,
  target.consignor_id = source.consignor_id,
  target.lot_number = source.lot_number,
  target.quantity = source.quantity,
  target.startingbid = source.startingbid,
  target.lowestimate = source.lowestimate,
  target.highestimate = source.highestimate,
  target.midestimate = source.midestimate,
  target.hammerprice = source.hammerprice,
  target.commissionamount = source.commissionamount,
  target.buyerspremium = source.buyerspremium,
  target.pricerealized = source.pricerealized,
  target.favoritescount = source.favoritescount,
  target.start_date = source.start_date,
  target.end_date = source.end_date,
  target.sold_flag = source.sold_flag,
  target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (
  user_dim_key,lot_dim_key,consignor_dim_key, auction_id, user_id, consignor_id, lot_id, lot_number,
  quantity, startingbid, lowestimate, highestimate, midestimate,
  hammerprice, commissionamount, buyerspremium, pricerealized,
  favoritescount,start_date, end_date, sold_flag, created_at, updated_at
)
VALUES (
  source.user_dim_key,source.lot_dim_key,source.consignor_dim_key, source.auction_id, source.user_id, source.consignor_id,
  source.lot_id, source.lot_number, source.quantity, source.startingbid,
  source.lowestimate, source.highestimate, source.midestimate,
  source.hammerprice, source.commissionamount, source.buyerspremium,
  source.pricerealized, source.favoritescount,source.start_date,source.end_date, source.sold_flag,
  current_timestamp(), current_timestamp()
)

---
## Step 4 · Verification

Queries the final `company_db.gold.auction_results_fact` table to validate the output. Spot-check that:
- Row count matches expected lot results
- `sold_flag` distribution looks reasonable (sold vs unsold ratio)
- `user_dim_key` is non-null for sold lots (indicates successful dimension join)
- `pricerealized` is non-zero for all rows where `sold_flag = 1`
- `created_at` and `updated_at` are populated for all rows

In [ ]:
%sql
-- DESCRIBE HISTORY company_db.gold.auction_results_fact;
-- RESTORE TABLE company_db.gold.auction_results_fact TO VERSION AS OF 8;

In [ ]:
%sql

CREATE OR REPLACE VIEW company_db.gold.auction_results_superset_view
AS
(
select 
ar.auction_id,
ar.lot_id,
ar.consignor_id,
ar.consignor_name,
ar.consignor_email,
ar.lot_number,
ar.item_name,
ar.country,
ar.region,
ar.item_year,
ar.format,
ar.status,
ar.lot_category,
ar.quantity,
ar.lowestimate,
ar.highestimate,
ar.midestimate,
concat('$',ar.lowestimate,' - ', '$',ar.highestimate) as `est.range`,
ar.hammerprice,
ar.pricerealized,
ar.start_date,
ar.end_date,
MONTHNAME(ar.start_date) AS month_name,
quarter(ar.start_date) AS quarter,
MONTH(ar.start_date) AS month_number,
YEAR(ar.start_date) AS year,
CASE
WHEN hammerprice > highestimate THEN 'Above High'
WHEN hammerprice < lowestimate THEN 'Below High'
ELSE
'In Range'
END AS estimates
from 
company_db.silver.auctionresults as ar 
left join company_db.silver.winningbidders as wb
ON wb.lot_number = ar.lot_number 
AND 
wb.auction_id = ar.auction_id
LEFT JOIN company_db.gold.lots_dim as dl
ON (ar.start_date = dl.start_date AND ar.lot_number = dl.lot_number) 
LEFT JOIN company_db.gold.consignor_dim as cm
ON cm.consignor_id = ar.consignor_id
-- where 
-- dl.auction_category = 'collectibles'
)

In [ ]:
# %sql
# SELECT 
# *
# FROM
# company_db.gold.auction_results_superset_view